BCH coding: BCH(31,21,5)

- to correct upto t=2 errors dmin = 5

- degree(gen) = 10 found using cyclotomic sets

- k = n - deg(gen) = 31 - 10 = 21

In [1]:
import numpy as np
import galois
import random
import math
import itertools

In [2]:
m = 5
n = 2**m - 1
t = 2

gf = galois.GF(2**m)
print(gf.properties)
alpha = gf.primitive_element
print(f'Primitive element: {alpha}')

Galois Field:
  name: GF(2^5)
  characteristic: 2
  degree: 5
  order: 32
  irreducible_poly: x^5 + x^2 + 1
  is_primitive_poly: True
  primitive_element: x
Primitive element: 2


In [3]:
x = galois.Poly.Identity(galois.GF2)
x_n = x**n + galois.GF2(1)

factors_32, _ = galois.factors(x_n)

for i in factors_32:
    print(i)


x + 1
x^5 + x^2 + 1
x^5 + x^3 + 1
x^5 + x^3 + x^2 + x + 1
x^5 + x^4 + x^2 + x + 1
x^5 + x^4 + x^3 + x + 1
x^5 + x^4 + x^3 + x^2 + 1


In [4]:
def con_coset(n):
    cyclotomic_coset = {}
    for i in range(0,n):
        # if i not in cyclotomic_coset
        leader = i
        if not any(i in array for array in cyclotomic_coset.values()):
            temp = np.array([])
            while i not in temp:
                temp = np.append(temp, int(i))
                i *= 2
                i %= n
            cyclotomic_coset[i] = temp

    return cyclotomic_coset

In [5]:
coset = con_coset(n)
print(coset)

{0: array([0.]), 1: array([ 1.,  2.,  4.,  8., 16.]), 3: array([ 3.,  6., 12., 24., 17.]), 5: array([ 5., 10., 20.,  9., 18.]), 7: array([ 7., 14., 28., 25., 19.]), 11: array([11., 22., 13., 26., 21.]), 15: array([15., 30., 29., 27., 23.])}


In [6]:
bch = galois.BCH(n, n-t*m)
print(bch)

BCH Code:
  [n, k, d]: [31, 21, 5]
  field: GF(2)
  extension_field: GF(2^5)
  generator_poly: x^10 + x^9 + x^8 + x^6 + x^5 + x^3 + 1
  is_primitive: True
  is_narrow_sense: True
  is_systematic: True


In [7]:
msg = galois.GF2.Random(21)
print(f"Message Chosen: {msg}")

encoded = bch.encode(msg)
print(f"Encoded Message: {encoded}")

decoded = bch.decode(encoded)
print(f"Decoded is {np.all(msg == decoded)}")

Message Chosen: [0 0 1 0 1 0 1 0 1 1 1 0 1 0 0 1 1 0 1 0 0]
Encoded Message: [0 0 1 0 1 0 1 0 1 1 1 0 1 0 0 1 1 0 1 0 0 0 1 0 0 1 1 1 0 0 1]
Decoded is True


In [8]:
con_primitive = [i for i in range(1,2*t+1)]
print(f'Powers of consecutive primitive elements selected: {con_primitive}')

coset_iden = np.array([])
for i in con_primitive:
    key = [keys for keys, arr in coset.items() if i in arr]
    coset_iden = np.append(coset_iden, key)
coset_iden = np.unique(coset_iden)
print(coset_iden)

gout = galois.GF2(1)
for i in coset_iden:
    gout *= gf.minimal_poly(alpha**int(i))

print(f'Generator Polynomial: {gout}')
l_gout = len(gout.coeffs)
gout_coeff = np.append(gout.coeffs[::-1],[0]*(n-l_gout))
print(len(gout_coeff))

Powers of consecutive primitive elements selected: [1, 2, 3, 4]
[1. 3.]
Generator Polynomial: x^10 + x^9 + x^8 + x^6 + x^5 + x^3 + 1
31


In [9]:
Gout = np.array(gout_coeff)
for i in range(1,n-gout.degree):
    row = np.roll(gout_coeff, i)

    Gout = np.vstack((Gout, row))

print(Gout)

[[1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0

In [10]:
# systematic matrix
G_sys = galois.FieldArray.row_reduce(Gout)
print(G_sys.shape)

(21, 31)


In [11]:
hout = x_n // gout
print(f'Parity-check polynomial for BCH code: {hout}')
r = len(hout.coeffs)
hout_coeff = np.append(hout.coeffs[::-1], [0]*(n-r))
print(hout_coeff)

Parity-check polynomial for BCH code: x^21 + x^20 + x^18 + x^16 + x^14 + x^13 + x^12 + x^11 + x^8 + x^5 + x^3 + 1
[1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0]


In [12]:
Hout = np.array(hout_coeff)
for i in range(1, n-r+1):
    row = np.roll(hout_coeff, i)
    Hout = np.vstack((Hout, row))

print(Hout)

[[1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1]]


In [13]:
H_sys = galois.FieldArray.row_reduce(Hout)
print(H_sys)

[[1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1]
 [0 1 0 0 0 0 0 0 0 0 1 1 0 1 1 1 1 0 1 1 0 1 0 0 0 1 1 1 1 1 1]
 [0 0 1 0 0 0 0 0 0 0 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 0 0 1 0 1 0]
 [0 0 0 1 0 0 0 0 0 0 0 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 0 0 1 0 1]
 [0 0 0 0 1 0 0 0 0 0 1 0 1 0 1 0 1 0 0 1 1 0 0 0 1 1 0 0 1 1 1]
 [0 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 1 0 1 0 1 1 1 1 0 0 1 1 0]
 [0 0 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 1 0 1 0 1 1 1 1 0 0 1 1]
 [0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1]]


Gin Construction

In [14]:
gout_fac, _ = galois.Poly.factors(gout)
# print(gout_fac)
gin_choice = [i for i in factors_32[1:] if i not in gout_fac]
gin = random.choice(gin_choice)
print(f'Choosen innercode polynomial: {gin}')

Choosen innercode polynomial: x^5 + x^3 + 1


In [15]:
k = n - t * m
gin_rotate = k - m

gin_coeff = np.append(gin.coeffs, [0]*(gin_rotate-1))
Gin = np.array(gin_coeff)
for i in range(1, gin_rotate):
    row = np.roll(gin_coeff, i)
    Gin = np.vstack((Gin, row))

print(Gin.shape)

(16, 21)


In [16]:
g = gin * gout
print(g)

g_coeff = np.append(g.coeffs, [0]*(n-g.degree-1))
# print(len(g_coeff))
# print(n-g.degree-1)
G = np.array(g_coeff)
for i in range(1, n-g.degree):
    row = np.roll(g_coeff, i)
    G = np.vstack((G, row))

print(G)

x^15 + x^14 + x^12 + x^8 + 1
[[1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1

In [17]:
G_sys = galois.FieldArray.row_reduce(G)
print(G_sys)

[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 1 1 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 1 1 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 1 1 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 1 1 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 1 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 0 0 1 1 0 0 0 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0

In [18]:
h = x_n // g

print(f'hin polynomial: {h}')
h_coeff = np.append(h.coeffs[::-1], [0]*(n-h.degree-1))
# print(len(hout_coeff))
H = np.array(h_coeff)
for i in range(1, n-h.degree):
    row = np.roll(h_coeff, i)
    H = np.vstack((H, row))

print(H)

hin polynomial: x^16 + x^15 + x^14 + x^12 + x^8 + 1
[[1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 

In [19]:
H_sys = galois.FieldArray.row_reduce(H)
print(H)

[[1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 1]]


In [20]:
print(G @ H.T)

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


Si(x) Implementation

In [21]:
print(g)    
B = k-m
print(f'The length of message sequence: {B}')

x^15 + x^14 + x^12 + x^8 + 1
The length of message sequence: 16


In [44]:
S = {}
Q = {}
for i in range(1, B+1):
    s = x**(n-i) % g
    q = x**(n-i) // g
    S[i] = s
    Q[i] = q
S = dict(sorted(S.items(), reverse=True))
print(S)

{16: Poly(x^14 + x^12 + x^8 + 1, GF(2)), 15: Poly(x^14 + x^13 + x^12 + x^9 + x^8 + x + 1, GF(2)), 14: Poly(x^13 + x^12 + x^10 + x^9 + x^8 + x^2 + x + 1, GF(2)), 13: Poly(x^14 + x^13 + x^11 + x^10 + x^9 + x^3 + x^2 + x, GF(2)), 12: Poly(x^11 + x^10 + x^8 + x^4 + x^3 + x^2 + 1, GF(2)), 11: Poly(x^12 + x^11 + x^9 + x^5 + x^4 + x^3 + x, GF(2)), 10: Poly(x^13 + x^12 + x^10 + x^6 + x^5 + x^4 + x^2, GF(2)), 9: Poly(x^14 + x^13 + x^11 + x^7 + x^6 + x^5 + x^3, GF(2)), 8: Poly(x^7 + x^6 + x^4 + 1, GF(2)), 7: Poly(x^8 + x^7 + x^5 + x, GF(2)), 6: Poly(x^9 + x^8 + x^6 + x^2, GF(2)), 5: Poly(x^10 + x^9 + x^7 + x^3, GF(2)), 4: Poly(x^11 + x^10 + x^8 + x^4, GF(2)), 3: Poly(x^12 + x^11 + x^9 + x^5, GF(2)), 2: Poly(x^13 + x^12 + x^10 + x^6, GF(2)), 1: Poly(x^14 + x^13 + x^11 + x^7, GF(2))}


In [45]:
P = []
for i, s in S.items():
    # print(f'{i} --> {s}')
    coeffs = s.coeffs[::-1]

    row = np.zeros(B-1, dtype=int)

    row[:len(coeffs)] = coeffs
    P.append(row)
P = np.array(P)
print(P)


[[1 0 0 0 0 0 0 0 1 0 0 0 1 0 1]
 [1 1 0 0 0 0 0 0 1 1 0 0 1 1 1]
 [1 1 1 0 0 0 0 0 1 1 1 0 1 1 0]
 [0 1 1 1 0 0 0 0 0 1 1 1 0 1 1]
 [1 0 1 1 1 0 0 0 1 0 1 1 0 0 0]
 [0 1 0 1 1 1 0 0 0 1 0 1 1 0 0]
 [0 0 1 0 1 1 1 0 0 0 1 0 1 1 0]
 [0 0 0 1 0 1 1 1 0 0 0 1 0 1 1]
 [1 0 0 0 1 0 1 1 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 1 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 1 0 1 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 1 0 1 1 0 0 0 0]
 [0 0 0 0 1 0 0 0 1 0 1 1 0 0 0]
 [0 0 0 0 0 1 0 0 0 1 0 1 1 0 0]
 [0 0 0 0 0 0 1 0 0 0 1 0 1 1 0]
 [0 0 0 0 0 0 0 1 0 0 0 1 0 1 1]]


K = 15 not 10( gin * gout)

P --> 16 x 15 (n-k, k)

G_syst --> 16 x 31 (n-k, n)

H_syst --> 15 x 31 (k, n)

In [46]:
I_16 = np.eye(B)
# print(I_16.shape)
# print(I_16)
G_syst = np.hstack((-P,I_16)).astype(int)%2  #  over GF(2)
G_syst = galois.GF2(G_syst)
print(G_syst)

[[1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 0 0 0 1 1 0 0 1 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 1 1 1 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 1 0 0 0 0 0 1 1 1 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 1 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 1 1 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 1 1 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 1 1 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 0 1

In [47]:
I_15 = np.eye(n-B)
H_syst = np.hstack((I_15, P.T)).astype(int)
H_syst = galois.GF2(H_syst)
print(H_syst)

[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 1 1 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 1 1 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 1 1 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 1 1 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 1 0 0 1 1 0 0 0 0 0 0 1 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 1 0 0 1 1 0 0 0 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1]]


In [48]:
check = (G_syst @ H_syst.T )
print(check)

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


In [50]:
for i in range(1, B+1):
    l = g * Q[i] + S[i]
    print(l)

x^30
x^29
x^28
x^27
x^26
x^25
x^24
x^23
x^22
x^21
x^20
x^19
x^18
x^17
x^16
x^15


In [51]:
print(gout)
print(gout_coeff)
print(Gout)
# e1 = np.zeros(k)
# e1[0] = 1
e1 = galois.GF2.Zeros(k)
e1[0] = 1
print(f'Affine translation vector: {e1}')

x^10 + x^9 + x^8 + x^6 + x^5 + x^3 + 1
[1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[[1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 0 

In [52]:
# e1 = galois.FieldArray.Vector(e1)
e_gout = e1 @ Gout
print(f"Vector to be used for affine translation: {e_gout}")

Vector to be used for affine translation: [1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [53]:
msg = galois.GF2.Random(B)
print(f"Message to be transmitted: {msg}")

Message to be transmitted: [1 0 0 1 0 1 1 1 0 0 1 1 1 1 1 0]


In [54]:
codeword = msg @ G_syst + e_gout
print(codeword)

[0 0 1 1 1 1 1 0 0 0 0 1 1 0 1 1 0 0 1 0 1 1 1 0 0 1 1 1 1 1 0]


In [55]:
c_hat = codeword - e_gout
syndrome = c_hat@ H_syst.T
print(f"Syndrome: {syndrome}")
print(c_hat[-B:])

Syndrome: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[1 0 0 1 0 1 1 1 0 0 1 1 1 1 1 0]


In [56]:
print(hout)
print(hout_coeff)
print(Hout)

x^21 + x^20 + x^18 + x^16 + x^14 + x^13 + x^12 + x^11 + x^8 + x^5 + x^3 + 1
[1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0]
[[1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 1 0 1 0 1 0 1 1]]


Similar to Si(x) for Gout and Hout (systematic matrices)

In [57]:
print(f'The length of inner code-word: {k}')
S_out ={}
for i in range(1, k+1):
    s_out = x**(n-i) % gout 
    S_out[i] = s_out

S_out = dict(sorted(S_out.items(), reverse=True))
print(S_out)

The length of inner code-word: 21
{21: Poly(x^9 + x^8 + x^6 + x^5 + x^3 + 1, GF(2)), 20: Poly(x^8 + x^7 + x^5 + x^4 + x^3 + x + 1, GF(2)), 19: Poly(x^9 + x^8 + x^6 + x^5 + x^4 + x^2 + x, GF(2)), 18: Poly(x^8 + x^7 + x^2 + 1, GF(2)), 17: Poly(x^9 + x^8 + x^3 + x, GF(2)), 16: Poly(x^8 + x^6 + x^5 + x^4 + x^3 + x^2 + 1, GF(2)), 15: Poly(x^9 + x^7 + x^6 + x^5 + x^4 + x^3 + x, GF(2)), 14: Poly(x^9 + x^7 + x^4 + x^3 + x^2 + 1, GF(2)), 13: Poly(x^9 + x^6 + x^4 + x + 1, GF(2)), 12: Poly(x^9 + x^8 + x^7 + x^6 + x^3 + x^2 + x + 1, GF(2)), 11: Poly(x^7 + x^6 + x^5 + x^4 + x^2 + x + 1, GF(2)), 10: Poly(x^8 + x^7 + x^6 + x^5 + x^3 + x^2 + x, GF(2)), 9: Poly(x^9 + x^8 + x^7 + x^6 + x^4 + x^3 + x^2, GF(2)), 8: Poly(x^7 + x^6 + x^4 + 1, GF(2)), 7: Poly(x^8 + x^7 + x^5 + x, GF(2)), 6: Poly(x^9 + x^8 + x^6 + x^2, GF(2)), 5: Poly(x^8 + x^7 + x^6 + x^5 + 1, GF(2)), 4: Poly(x^9 + x^8 + x^7 + x^6 + x, GF(2)), 3: Poly(x^7 + x^6 + x^5 + x^3 + x^2 + 1, GF(2)), 2: Poly(x^8 + x^7 + x^6 + x^4 + x^3 + x, GF(2)), 1

In [58]:
P_out = []
for i, s_out in S_out.items():
    row = np.zeros(n-k, dtype=int)
    coef = s_out.coeffs

    row[:len(coef)] = coef[::-1]

    P_out.append(row)
P_out = np.array(P_out)
print(P_out.shape)

# print(P_out)
# P_out = galois.GF2(P_out)
# print(P_out)

(21, 10)


In [59]:
i = np.eye(k)
print(i.shape)
G_out_syst = np.hstack((-P_out, i)).astype(int) % 2
# print(type(G_out))
G_out_syst = galois.GF2(G_out_syst)
print(G_out_syst)

(21, 21)
[[1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 1 1 1 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 0 1 1 1 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 1 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 1 1 1 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 1 1 1 1 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 1 1 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 1 0 0 1 1 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 0 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 1 0 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 1 1 1 0 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 1 

In [60]:
H_out_syst = np.hstack((np.eye(n-k), P_out.T)).astype(int)
H_out_syst = galois.GF2(H_out_syst)
print(H_out_syst)

[[1 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 1 1 1 1 0 0 1 0 0 1 0 1 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 1 1 1 1 0 0 1 0 0 1 0 1 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 1 1 1 1 0 0 1 0 0 1 0 1]
 [0 0 0 1 0 0 0 0 0 0 1 1 0 0 1 1 1 1 0 1 0 1 1 0 0 0 0 0 1 1 0]
 [0 0 0 0 1 0 0 0 0 0 0 1 1 0 0 1 1 1 1 0 1 0 1 1 0 0 0 0 0 1 1]
 [0 0 0 0 0 1 0 0 0 0 1 1 1 0 0 1 1 0 0 0 1 1 0 0 1 0 1 0 1 0 1]
 [0 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 1 0 1 1 1 1 1 1 0 1 1 1 1 1 0]
 [0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 1 0 1 1 1 1 1 1 0 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 1 0 0 0 1 0 1 1 0 1 1 1 1 0 1 1]
 [0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 1 1 1 1 0 0 1 0 0 1 0 1 0 0 1]]


In [61]:
print(H_out_syst[:,0])
print(H_out_syst[:,30])

[1 0 0 0 0 0 0 0 0 0]
[0 0 1 0 1 1 0 1 1 1]


In [62]:
print(G_out_syst @ H_out_syst.T)

[[0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]


Generation of Syndrome Table (look-up table)

In [63]:
syndrome_table = {}

GF2 = galois.GF2

e = GF2.Zeros(n)
s = tuple( (e @ H_out_syst.T).tolist() )
print((e @ H_out_syst.T).shape)
syndrome_table[s] = e
print(e)
print(syndrome_table)

for i in range(n):
    e = GF2.Zeros(n)
    e[i] = 1

    s = tuple( (e @ H_out_syst.T).tolist() )
    syndrome_table[s] = e

for i, j in itertools.combinations(range(n), 2):
    e = GF2.Zeros(n)
    e[i] = 1
    e[j] = 1

    s = tuple( (e @ H_out_syst.T).tolist() )

    if s == (0,)*10:
        print("Found zero syndrome!")
        print(i,j)
        print(e)

    syndrome_table[s] = e

(10,)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
{(0, 0, 0, 0, 0, 0, 0, 0, 0, 0): GF([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0], order=2)}


In [64]:
count = 0
for i, j in itertools.combinations(range(n), 2):
    # print(f'{i} -- {j}')
    count += 1

print(count)

465


In [65]:
print(syndrome_table[(0, 0, 0, 0, 0, 0, 0, 0, 0, 0)])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [66]:
z = x ** 30 + GF2(1)
print(z // gout)
print(z % gout)

x^20 + x^19 + x^17 + x^15 + x^13 + x^12 + x^11 + x^10 + x^7 + x^4 + x^2
x^9 + x^8 + x^7 + x^5 + x^4 + x^2 + 1
